# 第18章　风险管理系统

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch18_risk_management.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch18_risk_management.ipynb)

复现例18.1-18.4（参数/历史/蒙特卡洛 VaR、CVaR、情景分析）与图18-1，全书工具的风险集成。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi import var as v
from fi import plotting
plotting.use_chinese_style()


## 例18.1　参数法 VaR / CVaR（组合1亿, 久期5, 日波动5bp）


In [ ]:
sigma = v.bond_pnl_sigma(value=1e8, duration=5, yield_vol=0.0005)
print(f'日损益标准差 = {sigma:,.0f} 元')
for a in (0.95, 0.99):
    print(f'{a:.0%} 1日 VaR = {v.parametric_var(sigma, a):,.0f}   CVaR = {v.parametric_cvar(sigma, a):,.0f}')


## 例18.2　三种方法对比（正态 vs 厚尾）


In [ ]:
rng = np.random.default_rng(7)
fat = rng.standard_t(4, 100000) * sigma / np.sqrt(4/2)   # 厚尾损益
print(f'参数法(正态)   99% VaR = {v.parametric_var(sigma, 0.99):,.0f}')
mc_var, mc_cvar = v.monte_carlo_var(sigma, 0.99, n=200000, seed=1)
print(f'蒙特卡洛(正态) 99% VaR = {mc_var:,.0f}  CVaR = {mc_cvar:,.0f}')
print(f'历史法(厚尾)   99% VaR = {v.historical_var(fat, 0.99):,.0f}  CVaR = {v.historical_cvar(fat, 0.99):,.0f}  <- 厚尾更大')


## 图18-1　损益分布与 VaR/CVaR（编程实验 7）


In [ ]:
normal = rng.normal(0, sigma, 100000)
var99 = v.parametric_var(sigma, 0.99); cvar99 = v.parametric_cvar(sigma, 0.99)
fig, ax = plotting.new_axes()
ax.hist(normal/1e4, bins=120, density=True, alpha=0.5, label='正态损益')
ax.hist(fat/1e4, bins=200, density=True, alpha=0.4, label='厚尾损益')
ax.axvline(-var99/1e4, color='C3', ls='--', label=f'99% VaR≈{var99/1e4:.0f}万')
ax.axvline(-cvar99/1e4, color='C1', ls=':', label=f'99% CVaR≈{cvar99/1e4:.0f}万')
ax.set_xlim(-150, 150); ax.set_xlabel('日损益（万元）'); ax.set_ylabel('密度')
ax.set_title('图18-1　组合损益分布与 VaR/CVaR'); ax.legend()
fig.tight_layout()


## 例18.3-18.4　情景分析与压力测试


In [ ]:
print('利率情景（组合1亿, 久期5, 凸性30）:')
for dy, label in [(0.01, '+100bp'), (0.02, '+200bp'), (-0.01, '-100bp')]:
    print(f'  {label}: 组合损益 = {v.scenario_pnl(1e8, 5, 30, dy):,.0f} 元')
print('  注：+200bp 损失 < +100bp 的两倍 —— 正凸性减轻大幅上行的损失')


---

> 全书收尾：VaR/CVaR 度量正常市场风险（正态低估厚尾），压力测试审视极端情景；
> 风险系统是全书工具的集成——把定价、久期、曲线、信用、衍生品装进『市场变坏会怎样』的框架。
